<a href="https://colab.research.google.com/github/aMDy0k/workspace/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###### import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sympy import *
import requests
import json
import time
import sys
import os
import ast

from google.colab import drive
drive.mount('/content/drive')

###### словарь

In [ ]:
heroes = {
    1: "Anti-Mage", 2: "Axe", 3: "Bane", 4: "Bloodseeker", 5: "Crystal Maiden",
    6: "Drow Ranger", 7: "Earthshaker", 8: "Juggernaut", 9: "Mirana", 10: "Morphling",
    11: "Shadow Fiend", 12: "Phantom Lancer", 13: "Puck", 14: "Pudge", 15: "Razor",
    16: "Sand King", 17: "Storm Spirit", 18: "Sven", 19: "Tiny", 20: "Vengeful Spirit",
    21: "Windranger", 22: "Zeus", 23: "Kunkka", 25: "Lina", 26: "Lion",
    27: "Shadow Shaman", 28: "Slardar", 29: "Tidehunter", 30: "Witch Doctor", 31: "Lich",
    32: "Riki", 33: "Enigma", 34: "Tinker", 35: "Sniper", 36: "Necrophos",
    37: "Warlock", 38: "Beastmaster", 39: "Queen of Pain", 40: "Venomancer", 41: "Faceless Void",
    42: "Wraith King", 43: "Death Prophet", 44: "Phantom Assassin", 45: "Pugna", 46: "Templar Assassin",
    47: "Viper", 48: "Luna", 49: "Dragon Knight", 50: "Dazzle", 51: "Clockwerk",
    52: "Leshrac", 53: "Nature's Prophet", 54: "Lifestealer", 55: "Dark Seer", 56: "Clinkz",
    57: "Omniknight", 58: "Enchantress", 59: "Huskar", 60: "Night Stalker", 61: "Broodmother",
    62: "Bounty Hunter", 63: "Weaver", 64: "Jakiro", 65: "Batrider", 66: "Chen",
    67: "Spectre", 68: "Ancient Apparition", 69: "Doom", 70: "Ursa", 71: "Spirit Breaker",
    72: "Gyrocopter", 73: "Alchemist", 74: "Invoker", 75: "Silencer", 76: "Outworld Destroyer",
    77: "Lycan", 78: "Brewmaster", 79: "Shadow Demon", 80: "Lone Druid", 81: "Chaos Knight",
    82: "Meepo", 83: "Treant Protector", 84: "Ogre Magi", 85: "Undying", 86: "Rubick",
    87: "Disruptor", 88: "Nyx Assassin", 89: "Naga Siren", 90: "Keeper of the Light", 91: "Io",
    92: "Visage", 93: "Slark", 94: "Medusa", 95: "Troll Warlord", 96: "Centaur Warrunner",
    97: "Magnus", 98: "Timbersaw", 99: "Bristleback", 100: "Tusk", 101: "Skywrath Mage",
    102: "Abaddon", 103: "Elder Titan", 104: "Legion Commander", 105: "Techies", 106: "Ember Spirit",
    107: "Earth Spirit", 108: "Underlord", 109: "Terrorblade", 110: "Phoenix", 111: "Oracle",
    112: "Winter Wyvern", 113: "Arc Warden", 114: "Monkey King", 119: "Dark Willow", 120: "Pangolier",
    121: "Grimstroke", 123: "Hoodwink", 126: "Void Spirit", 128: "Snapfire", 129: "Mars",
    131: "Muerta", 135: "Dawnbreaker", 136: "Marci", 137: "Primal Beast",
    138: "Ringmaster", 145: "Kez", 155: "Largo"
}

### data

In [ ]:
def clean_teams_in_row(row):
    def parse_value(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except:
                try:
                    return ast.literal_eval(val)
                except:
                    return [
                        int(x)
                        for x in val.split(",")
                        if x.strip().isdigit()
                    ]
        return [int(x) for x in val] if isinstance(val, list) else []

    row["radiant_team"] = parse_value(row.get("radiant_team", []))
    row["dire_team"] = parse_value(row.get("dire_team", []))
    return row

In [ ]:
def parser():
    global df

    # Проверяем, существует ли глобальный df в памяти ноутбука
    if "df" in globals() and not df.empty:
        # Так как match_id теперь в индексе, берем минимум прямо из индекса
        min_id = df.index.min()

        # Проверяем, что минимум — это реальное число
        if pd.notna(min_id) and min_id > 1000000000:
            less_than_match_id = int(min_id)
            print(
                f"Парсер продолжит сбор, начиная с match_id < {less_than_match_id}"
            )
        else:
            less_than_match_id = None
            print("Парсер начинает сбор с самых актуальных матчей.")
    else:
        less_than_match_id = None
        print("Парсер начинает сбор с самых актуальных матчей.")

    # ---------------------------

    # Базовый URL для получения СЫРЫХ публичных матчей
    url = "https://api.opendota.com/api/publicMatches"
    all_raw_matches = []

    print("Начинаем выкачивать сырой датасет напрямую...")

    for i in range(50):
        params = {}
        if less_than_match_id:
            params["less_than_match_id"] = less_than_match_id

        try:
            response = requests.get(url, params=params)

            if response.status_code == 200:
                batch = response.json()
                if not batch:
                    print("Матчи закончились.")
                    break

                all_raw_matches.extend(batch)
                print(
                    f"Скачана пачка {i+1}. Всего матчей в датасете: {len(all_raw_matches)}"
                )

                less_than_match_id = batch[-1]["match_id"]
                time.sleep(1)
            else:
                print(
                    f"Ошибка. Статус: {response.status_code}, Ответ: {response.text}"
                )
                break
        except Exception as e:
            print(f"Ошибка сети: {e}")
            break

    if all_raw_matches:
        filename = "raw_matches_dataset.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(all_raw_matches, f, indent=4, ensure_ascii=False)

        print(f"\nСырой датасет сохранен в файл: '{filename}'")
    else:
        print("Не удалось собрать данные.")
        return  # Если данных нет, прерываем выполнение функции

    # ---------------------------

    if all_raw_matches:
        new_df = pd.DataFrame(all_raw_matches)

        if "match_id" in new_df.columns:
            new_df = new_df.set_index("match_id")

        # Проверяем наличие df в глобальной области видимости
        df = (
            pd.concat([df, new_df], axis=0)
            if ("df" in globals() and not df.empty)
            else new_df
        )

        df = df[~df.index.duplicated(keep="first")]
        df = df.apply(clean_teams_in_row, axis=1)
        df = df[(df["game_mode"] == 22) & (df["duration"] > (15 * 60))]

        print(
            f"📊 Датасет успешно увеличен! Текущий размер: {df.shape} уникальных матчей.\n"
        )
    else:
        print("Новых матчей не собрано.")

    # ------------------------------

    # Сохраняем, используя глобальную переменную path (убедитесь, что она объявлена в ноутбуке)
    df.to_csv(path, index=True, index_label="match_id")
    print("Файл успешно сохранен на Google Диск!")


In [ ]:
path = "/content/drive/MyDrive/core/df/dota2.csv"

if os.path.exists(path) and os.path.getsize(path) > 0:
    # Читаем файл и сразу ставим match_id как индекс таблицы
    df = pd.read_csv(path, index_col="match_id")
    print(f"📦 База успешно загружена с диска! Матчей: {len(df)}")
    df = df.apply(clean_teams_in_row, axis=1)
else:
    df = pd.DataFrame()
    print("🆕 База пуста, начинаем с нуля.")
    parser()

In [ ]:
def auto_collector(total_cycles=30, cooldown_minutes=2):
    """Автоматически запускает parser() в цикле с кулдауном.

    total_cycles: сколько раз вызвать parser() (30 циклов по 50 пачек = ~150к
    сырых матчей) cooldown_minutes: сколько минут отдыхать между вызовами
    """
    print(f"🚀 Автосборщик запущен!")
    print(
        f"Запланировано циклов: {total_cycles} | Пауза между ними: {cooldown_minutes} мин.\n"
    )

    for cycle in range(1, total_cycles + 1):
        print(
            f"=== 🔄 ЗАПУСК ЦИКЛА {cycle}/{total_cycles} ({time.strftime('%H:%M:%S')}) ==="
        )

        try:
            # Вызываем ваш готовый парсер
            parser()
        except Exception as e:
            # Защита от критического падения (например, если отвалился Google Диск или интернет)
            print(
                f"⚠️ Произошла ошибка во время работы парсера: {e}. Ждем кулдаун и пробуем снова."
            )

        if cycle == total_cycles:
            print("\n🎉 Все запланированные циклы успешно завершены!")
            break

        # Кулдаун с красивым таймером обратного отсчета в секундах
        cooldown_seconds = cooldown_minutes * 60
        print(f"\n💤 Цикл {cycle} завершен. Уходим на кулдаун...")

        for remaining in range(cooldown_seconds, 0, -1):
            mins, secs = divmod(remaining, 60)
            # \r позволяет обновлять строку на месте, не заспамливая лог экрана
            sys.stdout.write(
                f"\r⏳ До следующего сбора осталось: {mins:02d} мин {secs:02d} сек"
            )
            sys.stdout.flush()
            time.sleep(1)

        print("\n")  # Сдвиг строки после окончания таймера


# === ЗАПУСК АВТОСБОРА ===
# Запустит парсер 30 раз с паузой в 2 минуты между ними. Будет работать около 1.5 часов.
# auto_collector(total_cycles=30, cooldown_minutes=2)


In [ ]:
def i():
  print("=== ЭКСПРЕСС-ПРОВЕРКА ДАТАСЕТА ===")
  print(f" Всего матчей в таблице: {df.shape[0]}")
  print(f" Количество столбцов: {df.shape[1]}")

  # 1. Проверяем индекс
  is_match_id_index = df.index.name == 'match_id' or (df.index.astype(str).str.len() > 8).all()
  print(f" Индекс является 'match_id': {' SUCCESS' if is_match_id_index else '❌ FAILED (Индекс сбился)'}")
  print(f" Пример текущего индекса (первых 3 строки): {list(df.index[:3])}")

  # 2. Проверяем отсутствие дубликатов
  dup_count = df.index.duplicated().sum()
  print(f" Найденные дубликаты матчей: {dup_count} ({' SUCCESS' if dup_count == 0 else '❌ FAILED'})")

  # 3. Проверяем типы данных в командах
  first_rad = df['radiant_team'].iloc[0] if not df.empty else None
  is_clean_list = isinstance(first_rad, list) and len(first_rad) > 0 and isinstance(first_rad[0], int)
  print(f" Формат героев в ячейках: {' SUCCESS (Чистые списки чисел)' if is_clean_list else '❌ FAILED (Всё еще текст)'}")

  # 4. Проверяем среднее количество героев через правильный метод .apply(len)
  if is_clean_list:
      rad_len = df['radiant_team'].apply(len).mean()
      dire_len = df['dire_team'].apply(len).mean()
      print(f" Среднее количество героев в матче: Radiant={rad_len:.2f}, Dire={dire_len:.2f} ({' SUCCESS' if rad_len == 5.0 and dire_len == 5.0 else '⚠️ Внимание, есть неполные составы!'})")

  # 5. Проверяем фильтры режима и времени
  min_duration = df['duration'].min() / 60
  unique_modes = df['game_mode'].unique()
  print(f" Минимальная длительность матча: {min_duration:.1f} мин ({' SUCCESS' if min_duration >= 15 else '❌ Есть слишком короткие матчи'})")
  print(f" Уникальные режимы игры в базе: {unique_modes} ({' SUCCESS' if list(unique_modes) == [22] else '❌ Есть другие режимы кроме All Pick (22)'})")
  print("==================================")
i()

In [ ]:
len(df[df['radiant_win']]),len(df[df['radiant_win']==false])

In [ ]:
len(df[df['radiant_win']])/len(df['radiant_win'])*100

### ml

#####выборка

In [ ]:
arr = pd.concat([df['radiant_win'],df['radiant_team'],df['dire_team']],axis=1)\
.to_numpy()

In [ ]:
arr

In [ ]:
# 1. Извлекаем данные
y = arr[:, 0].astype(int)

# Превращаем колонки списков в плоские двумерные матрицы NumPy размера (7507, 5)
# Это делается мгновенно, так как длина всех списков строго равна 5
radiant_matrix = np.array(list(arr[:, 1]))
dire_matrix = np.array(list(arr[:, 2]))

# 2. Собираем все уникальные ID героев векторно
unique_heroes = np.unique(np.hstack([radiant_matrix, dire_matrix]))

# 3. Создаем пустую матрицу нулей строго под размер
num_matches = len(arr)
X_matrix = np.zeros((num_matches, len(unique_heroes)))

# Создаем векторный маппинг: переводим ID героев в индексы столбцов (от 0 до 136)
# np.searchsorted идеально находит, на каком месте должен стоять каждый ID героя
radiant_cols = np.searchsorted(unique_heroes, radiant_matrix)
dire_cols = np.searchsorted(unique_heroes, dire_matrix)

# 4. ВЕКТОРНАЯ МАГИЯ
# Создаем индекс строк для каждой из 5 позиций героев
row_indices = np.arange(num_matches)[:, np.newaxis]

# За один микрошаг заполняем всю матрицу единицами и минус единицами
X_matrix[row_indices, radiant_cols] = 1
X_matrix[row_indices, dire_cols] = -1

# 5. Оборачиваем в DataFrame
hero_cols = [f"{hero_id}" for hero_id in unique_heroes]
X = pd.DataFrame(X_matrix, columns=hero_cols)

print(f"Размер: {X.shape}")


In [ ]:
from sklearn.model_selection import train_test_split

# Разбиваем X и y на обучающую (Train) и проверочную (Test) выборки
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=42, stratify=y
)

# Параметр stratify=y как раз проследит, чтобы в обучающей и тестовой выборках
# сохранилась та самая пропорция 54% на 46%, которую мы обсудили!

print(f"Выборка успешно разделена!")
print(f"Размер Train (на чём учим): {X_train.shape}")
print(f"Размер Test (на чём проверяем): {X_test.shape}")


#####train

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

# Обучаем модель с балансировкой классов, судя строго по героям
model = LogisticRegression(
    C=0.001,
    penalty='l2',
    fit_intercept=False,
    class_weight='balanced',  # <-- ВОТ ОН, РЫЧАГ БАЛАНСА
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

# Делаем предсказания
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Считаем точность и строим матрицу
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Реально: Dire Win", "Реально: Radiant Win"],
    columns=["Модель: Dire Win", "Модель: Radiant Win"]
)

print("=== ИДЕАЛЬНО СБАЛАНСИРОВАННАЯ МАТРИЦА ОШИБОК ===")
print(cm_df)
print(f"\nТочность (Accuracy): {acc * 100:.2f}% ({acc:.4f})")

total_matches = cm.sum()
print(f"• Ошибка False Radiant: {cm[0][1]} раз ({cm[0][1]/total_matches*100:.2f}%)")
print(f"• Ошибка False Dire: {cm[1][0]} раз ({cm[1][0]/total_matches*100:.2f}%)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Генерируем сетку отступов от 0% до 3.5% с шагом 0.1% (0.001)
target_margins = np.arange(0.00, 0.099, 0.001)

print("🔬 ПОИСК ИДЕАЛЬНОЙ ТОЧКИ (Сетка микропорогов):\n")
print(
    f"{'Отступ':<8} | {'Порог Dire':<11} | {'Порог Rad':<11} | {'Взято матчей':<14} | {'Доля (%)':<10} | {'Accuracy':<10}"
)
print("-" * 76)

for margin in target_margins:
    low_p = 0.5 - margin
    high_p = 0.5 + margin

    # Фильтруем матчи, где модель перешагнула порог уверенности
    confident_mask = (y_prob >= high_p) | (y_prob <= low_p)
    y_test_sub = y_test[confident_mask]
    y_pred_sub = y_pred[confident_mask]

    # Если матчи по такому порогу нашлись, считаем метрики
    if len(y_test_sub) > 0:
        cm_sub = confusion_matrix(y_test_sub, y_pred_sub)
        correct_predictions = cm_sub.diagonal().sum()
        total_sub_matches = cm_sub.sum()

        current_acc = (correct_predictions / total_sub_matches) * 100
        share = (total_sub_matches / len(y_test)) * 100

        print(
            f"{margin:<8.3f} | {low_p:<11.3f} | {high_p:<11.3f} | {total_sub_matches:<14} | {share:<10.1f}% | {current_acc:.2f}%"
        )
    else:
        print(
            f"{margin:<8.3f} | {low_p:<11.3f} | {high_p:<11.3f} | {'0':<14} | 0.0%       | —"
        )


#####-|

In [ ]:
# 1. Извлекаем веса заново, чтобы убрать путаницу
hero_weights = pd.DataFrame(
    {"Герой_ID": X.columns, "Вес (Влияние на победу)": model.coef_[0]}
)

# Очищаем ID от приставки 'hero_'
hero_weights["Герой_ID"] = (
    hero_weights["Герой_ID"].str.replace("hero_", "").astype(int)
)

# Теперь самые большие плюсы гарантированно будут вверху, а минусы — внизу
hero_weights = hero_weights.sort_values(
    by="Вес (Влияние на победу)", ascending=False
).reset_index(drop=True)

# 2. Подвязываем имена из вашего словаря heroes
heroes_int_dict = {int(k): v for k, v in heroes.items()}
hero_weights["Имя героя"] = hero_weights["Герой_ID"].map(heroes_int_dict)

# Меняем порядок колонок для красоты
final_weights = hero_weights[
    ["Герой_ID", "Имя героя", "Вес (Влияние на победу)"]
]

print("🔥 НАСТОЯЩИЙ ТОП-10 ИМБ (Самый высокий винрейт за Radiant):")
print(final_weights.head(10).to_string(index=False))

print("\n🗑️ НАСТОЯЩИЙ ТОП-10 МУСОРА (Самый худший винрейт за Radiant):")
print(final_weights.tail(10).to_string(index=False))


In [ ]:
import numpy as np


def predict_pro_match(radiant_ids, dire_ids, heroes_dict, model_baseline):
    # 1. Принудительно приводим ключи словаря к int
    hd = {int(k): v for k, v in heroes_dict.items()}

    # 2. Вытаскиваем коэффициенты героев напрямую из нашей обученной модели
    # Нам больше не нужны промежуточные таблицы, берем данные из первоисточника
    feature_names = list(X.columns)
    weights = model_baseline.coef_[0]
    id_to_weight = {
        int(name.replace("hero_", "")): w
        for name, w in zip(feature_names, weights)
    }

    # 3. Считаем взвешенную сумму (модель с fit_intercept=False, поэтому стартуем с нуля)
    z = 0.0

    print("🎭 СОСТАВЫ КОМАНД:")
    r_names = [hd.get(h, f"Unknown ({h})") for h in radiant_ids]
    d_names = [hd.get(h, f"Unknown ({h})") for h in dire_ids]
    print(f"🟢 Radiant: {', '.join(r_names)}")
    print(f"🔴 Dire:    {', '.join(d_names)}\n")

    # Плюсуем веса Radiant, минусуем веса Dire
    for h_id in radiant_ids:
        z += id_to_weight.get(h_id, 0.0)
    for h_id in dire_ids:
        z -= id_to_weight.get(h_id, 0.0)

    # 4. Считаем чистую вероятность по Сигмоиде
    prob_radiant = 1 / (1 + np.exp(-z))

    # 5. Применяем наш доказанный порог уверенности (отступ 2.1%)
    margin = 0.021
    low_bound = 0.5 - margin
    high_bound = 0.5 + margin

    print("=== АНАЛИЗ ДРАФТА СИСТЕМOЙ ===")
    if low_bound < prob_radiant < high_bound:
        print(f"📊 Расчетный шанс Radiant: {prob_radiant * 100:.2f}%")
        print(
            "⚠️ СТАТУС: ПРОГНОЗ ОТМЕНЕН. Драфт слишком равный (уверенность модели < 2.1%)."
        )
    else:
        print("🔥 СТАТУС: ВЫДАНО УВЕРЕННОЕ ПРЕДСКАЗАНИЕ (Ожидаемая точность ~65.4%)")
        if prob_radiant >= high_bound:
            print(f"🏆 Прогноз на победу: 🟢 RADIANT (Шанс: {prob_radiant * 100:.2f}%)")
        else:
            print(f"🏆 Прогноз на победу: 🔴 DIRE (Шанс: {(1 - prob_radiant) * 100:.2f}%)")


# === ПРИМЕР ДЛЯ ТЕСТА ===
# Подставьте любые ID героев, чтобы проверить, пропустит ли модель матч или выдаст прогноз
my_radiant = [14, 11, 136, 64, 20]
my_dire = [119, 72, 12, 22, 51]

predict_pro_match(my_radiant, my_dire, heroes, model)
